<a href="https://colab.research.google.com/github/Timbo3399/INFM_SOTA2/blob/main/notebooks/workbook_timo_lotr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Woche 4: GPT-2 Small auf „Der Herr der Ringe“

**Autor:** Timo · **Datum:** 12.11.2026

Wir trainieren GPT-2 Small mit dem Text von *The Lord of the Rings* und vergleichen, wie verschiedene Modelle den Satzanfang **„The Elves are“** fortsetzen:

| Modell | Startpunkt | Training |
|---|---|---|
| Untrainiert | Zufallsgewichte | keins |
| LOTR von Grund auf | Zufallsgewichte | 3 Lernraten × 1 Epoche, dann 4 Epochen mit der besten |
| GPT-2 original | OpenAI-Gewichte | keins |
| GPT-2 nachtrainiert | OpenAI-Gewichte | 2 Epochen, Lernrate 5e-5 |
| Claude | – | Inkognito-Chat, per Hand |

Alle Entscheidungen und ihre Gründe sowie die Anleitung für Colab stehen in [`docs/lotr-training.md`](https://github.com/Timbo3399/INFM_SOTA2/blob/main/docs/lotr-training.md).
Der Code für Training, Messung und Textgenerierung liegt in `src/infm_sota2/` und hat Unittests. Dieses Notebook ruft ihn nur auf.

## 0. Einstellungen und Setup

`SMOKE_TEST = True` prüft die Pipeline lokal auf der CPU: winziges Modell, wenig Text, ohne W&B und ohne die OpenAI-Gewichte.
Zahlen und Texte aus diesem Modus haben keine Aussagekraft. Für den echten Lauf in Colab bleibt der Wert `False`.

In [ ]:
SMOKE_TEST = False

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "feature/lotr-training"  # nach dem Merge: "main"

if IN_COLAB:
    # Eigenes Paket installieren, Colabs torch/tiktoken/wandb bleiben unangetastet
    %pip install -q --no-deps "git+https://github.com/Timbo3399/INFM_SOTA2.git@{BRANCH}"

from importlib.metadata import requires, version

# Erwartete Versionen kommen aus pyproject.toml des installierten Pakets
for req in requires("infm-sota2") or []:
    name, _, wanted = req.partition("==")
    have = version(name).split("+")[0]
    if have != wanted:
        print(f"Achtung: {name} {have} installiert, pyproject.toml erwartet {wanted}")

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/INFM_SOTA2")
    RUN_DIR = Path("/content/runs")  # schnelle lokale Platte während des Trainings
else:
    BASE_DIR = Path.cwd()  # notebooks/
    RUN_DIR = BASE_DIR / "checkpoints" / "runs"

DATA_PATH = BASE_DIR / "data" / "The-lord-of-the-rings.txt"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"  # bleibt erhalten (in Colab auf Drive)
print("Text:", DATA_PATH)
print("Checkpoints:", CHECKPOINT_DIR)

Alle Werte, die das Ergebnis beeinflussen, stehen in `CONFIG`. Jeder W&B-Run bekommt die komplette `CONFIG` mit,
so ist später nachvollziehbar, womit er lief.

In [ ]:
import math

import torch
import wandb

from infm_sota2.model import GPT2_SMALL_CONFIG

CONFIG = {
    "seed": 123,
    "data": {
        "file": DATA_PATH.name,
        "start_line": "PROLOGUE",
        "end_line": "APPENDIX A",
        "book_title": "THE LORD OF THE RINGS",
        "train_ratio": 0.8,
        "val_ratio": 0.1,
        "max_chars": None,  # nur im SMOKE_TEST gesetzt
    },
    "seq_len": 256,
    "batch_size": 8,
    "weight_decay": 0.1,
    "grad_clip": 1.0,
    "eval_every": 100,
    "scratch_model": {**GPT2_SMALL_CONFIG, "context_length": 256},
    "lr_search": {"learning_rates": [1e-4, 4e-4, 1e-3], "epochs": 1},
    "main": {"epochs": 4},
    "finetune": {"learning_rate": 5e-5, "epochs": 2},
    "generation": {
        "prompt": "The Elves are",
        "samples": 3,
        "max_new_tokens": 40,
        "temperature": 0.8,
        "top_k": 40,
    },
}

if SMOKE_TEST:
    CONFIG["data"]["max_chars"] = {"train": 30_000, "val": 6_000, "test": 6_000}
    CONFIG |= {
        "seq_len": 32,
        "batch_size": 4,
        "eval_every": 5,
        "scratch_model": {
            **GPT2_SMALL_CONFIG,
            "context_length": 32,
            "emb_dim": 32,
            "n_heads": 2,
            "n_layers": 2,
        },
        "lr_search": {"learning_rates": [1e-3, 3e-3], "epochs": 1},
        "main": {"epochs": 1},
    }
    print("SMOKE_TEST: winziges Modell, gekürzter Text, Ergebnisse ohne Aussagekraft")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"  # Mixed Precision nur auf der GPU
CONFIG["environment"] = {
    "device": DEVICE.type,
    "gpu": torch.cuda.get_device_name() if USE_AMP else None,
    "torch": torch.__version__,
    "mixed_precision": USE_AMP,
    "branch": BRANCH,
    "colab": IN_COLAB,
}
WANDB_MODE = "disabled" if SMOKE_TEST else "online"
print(CONFIG["environment"])

In [ ]:
if not SMOKE_TEST:
    if IN_COLAB:
        from google.colab import userdata

        wandb.login(key=userdata.get("WANDB_API_KEY"))
    else:
        wandb.login()  # nutzt den Key aus `uv run wandb login`

## 1. Text bereinigen und aufteilen

Die Datei ist ein umgewandeltes Buch. `clean_book_text` behält nur den Erzähltext (Prolog bis Ende des letzten Kapitels),
entfernt Seitenkopfzeilen, fügt Sätze über Seitenwechsel und harte Zeilenumbrüche zusammen und löst Silbentrennung auf.
Jeder Schritt zählt mit, was er geändert hat. Die Tabelle der entfernten Kopfzeilen zeigt, ob die Regeln das Richtige treffen:
Dort sollten nur der Buchtitel, Kapitelnamen und Seitenzahlen stehen.

Danach wird der Text in drei zusammenhängende Blöcke geteilt: 80 % Training, 10 % Validierung (Auswahl von Lernrate und Checkpoint),
10 % Test (nur für den Vergleich am Ende). Aus urheberrechtlichen Gründen gibt das Notebook keine Ausschnitte des Buchs aus.

In [ ]:
from infm_sota2.data import clean_book_text, split_text

data_cfg = CONFIG["data"]
raw_text = DATA_PATH.read_text(encoding="utf-8")
text, stats = clean_book_text(
    raw_text,
    start_line=data_cfg["start_line"],
    end_line=data_cfg["end_line"],
    book_title=data_cfg["book_title"],
)

print(f"Zeichen roh: {len(raw_text):,}, bereinigt: {len(text):,}")
print(f"Zeilen im Erzählteil:          {stats.narrative_lines:,}")
print(f"Kopfzeilen entfernt:           {stats.headers_removed:,}")
print(f"Seitenwechsel zusammengefügt:  {stats.page_breaks_joined:,}")
print(f"Seitenwechsel als Absatz:      {stats.page_breaks_paragraph:,}")
print(f"Silbentrennungen aufgelöst:    {stats.hyphens_joined:,}")
print(f"Bindestriche behalten:         {stats.hyphens_kept:,}")
print(f"Absätze:                       {stats.paragraphs:,}")
print()
print("Entfernte Kopfzeilen nach Titel:")
for title, count in sorted(stats.header_titles.items(), key=lambda item: -item[1]):
    print(f"  {count:4d}  {title}")

In [ ]:
train_text, val_text, test_text = split_text(
    text, train_ratio=data_cfg["train_ratio"], val_ratio=data_cfg["val_ratio"]
)
if data_cfg["max_chars"] is not None:
    train_text = train_text[: data_cfg["max_chars"]["train"]]
    val_text = val_text[: data_cfg["max_chars"]["val"]]
    test_text = test_text[: data_cfg["max_chars"]["test"]]
    print("SMOKE_TEST: Texte gekürzt auf", data_cfg["max_chars"])

In [ ]:
from torch.utils.data import DataLoader

from infm_sota2.data import BPEDataset

seq_len = CONFIG["seq_len"]
datasets = {
    split: BPEDataset(split_text_, seq_len=seq_len, stride=seq_len)
    for split, split_text_ in [("train", train_text), ("val", val_text), ("test", test_text)]
}
tokenizer = datasets["train"].tokenizer
for split, dataset in datasets.items():
    print(f"{split:5s}: {len(dataset.tokens):>9,} Tokens, {len(dataset):>6,} Fenster à {seq_len}")


def make_train_loader() -> DataLoader:
    # Jeder Lauf bekommt einen neuen Loader mit gleichem Seed, also dieselbe Reihenfolge
    generator = torch.Generator().manual_seed(CONFIG["seed"])
    return DataLoader(
        datasets["train"],
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        drop_last=True,
        generator=generator,
    )


val_loader = DataLoader(datasets["val"], batch_size=CONFIG["batch_size"])
test_loader = DataLoader(datasets["test"], batch_size=CONFIG["batch_size"])
print(f"Schritte pro Epoche: {len(make_train_loader())}")

## 2. Gemeinsame Bausteine

Alle Modelle werden mit denselben Funktionen trainiert, gemessen und zum Schreiben gebracht.
So unterscheiden sich die Varianten nur in Startpunkt und Training.

- `record` misst den Test-Loss (mittlere Cross-Entropy pro Token, daraus Perplexity = e^Loss)
  und erzeugt drei Fortsetzungen von „The Elves are“ mit festen Seeds.
- `run_training` startet einen W&B-Run, trainiert und speichert den Checkpoint mit dem besten Validierungs-Loss.

In [ ]:
import shutil

from infm_sota2 import GPTModel
from infm_sota2.training import complete, evaluate, load_checkpoint, train

gen_cfg = CONFIG["generation"]
RESULTS: dict[str, dict] = {}


def new_scratch_model() -> GPTModel:
    torch.manual_seed(CONFIG["seed"])
    return GPTModel(CONFIG["scratch_model"]).to(DEVICE)


def sample_completions(model: GPTModel, context_size: int) -> list[str]:
    completions = []
    for i in range(gen_cfg["samples"]):
        generator = torch.Generator(device=DEVICE).manual_seed(CONFIG["seed"] + i)
        completions.append(
            complete(
                model,
                tokenizer,
                gen_cfg["prompt"],
                max_new_tokens=gen_cfg["max_new_tokens"],
                context_size=context_size,
                temperature=gen_cfg["temperature"],
                top_k=gen_cfg["top_k"],
                generator=generator,
            )
        )
    return completions


def record(name: str, model: GPTModel, context_size: int) -> None:
    test_loss = evaluate(model, test_loader, DEVICE, max_batches=None)
    RESULTS[name] = {
        "test_loss": test_loss,
        "perplexity": math.exp(test_loss),
        "completions": sample_completions(model, context_size),
    }
    print(f"{name}: Test-Loss {test_loss:.3f}, Perplexity {math.exp(test_loss):,.1f}")
    for i, completion in enumerate(RESULTS[name]["completions"], 1):
        print(f"  {i}. {gen_cfg['prompt']}{completion}")


def run_training(name, group, model, learning_rate, epochs, checkpoint_path):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=CONFIG["weight_decay"]
    )
    run = wandb.init(
        entity="hochschule-offenburg",
        project="SOTA_2",
        group=group,
        name=name,
        mode=WANDB_MODE,
        config={**CONFIG, "run": {"learning_rate": learning_rate, "epochs": epochs}},
    )
    result = train(
        model,
        optimizer,
        make_train_loader(),
        val_loader,
        DEVICE,
        epochs=epochs,
        eval_every=CONFIG["eval_every"],
        eval_batches=None,
        grad_clip=CONFIG["grad_clip"],
        use_amp=USE_AMP,
        checkpoint_path=checkpoint_path,
        checkpoint_metadata={"name": name, "learning_rate": learning_rate},
        log=lambda metrics, step: run.log(metrics, step=step),
    )
    run.summary["best_val_loss"] = result.best_val_loss
    run.summary["best_step"] = result.best_step
    run.finish()
    print(
        f"{name}: bester Validierungs-Loss {result.best_val_loss:.3f} "
        f"bei Schritt {result.best_step} von {result.steps}"
    )
    return result


def keep_checkpoint(path: Path) -> Path:
    # Von der lokalen Platte an den bleibenden Ort kopieren (in Colab: Drive)
    target = CHECKPOINT_DIR / path.name
    if target != path:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
    return target

## 3. Vorher: das untrainierte Modell

Ein Modell mit Zufallsgewichten rät, sein Loss sollte also bei ln(50257) ≈ 10,8 liegen.
Weicht er deutlich ab, stimmt etwas mit Daten, Zielen oder Loss nicht, und das Notebook bricht ab.

In [ ]:
untrained = new_scratch_model()
record("Untrainiert", untrained, CONFIG["scratch_model"]["context_length"])

expected_loss = math.log(CONFIG["scratch_model"]["vocab_size"])
untrained_loss = RESULTS["Untrainiert"]["test_loss"]
assert abs(untrained_loss - expected_loss) < 1.0, (
    f"Loss des untrainierten Modells {untrained_loss:.2f} weit weg von ln(V) = {expected_loss:.2f}"
)
del untrained

## 4. Lernraten-Suche: je 1 Epoche von Grund auf

Drei Läufe, die sich nur in der Lernrate unterscheiden (gleicher Seed, gleiche Daten-Reihenfolge).
Gewählt wird die Lernrate mit dem kleinsten Validierungs-Loss. Die Kurven stehen in W&B in der Gruppe `lotr-lr-search`.

In [ ]:
search_results = {}
for learning_rate in CONFIG["lr_search"]["learning_rates"]:
    result = run_training(
        name=f"lotr-scratch-lr{learning_rate:g}",
        group="lotr-lr-search",
        model=new_scratch_model(),
        learning_rate=learning_rate,
        epochs=CONFIG["lr_search"]["epochs"],
        checkpoint_path=None,
    )
    search_results[learning_rate] = result.best_val_loss

best_lr = min(search_results, key=search_results.__getitem__)
print()
for learning_rate, val_loss in search_results.items():
    marker = "  <- beste" if learning_rate == best_lr else ""
    print(f"Lernrate {learning_rate:g}: Validierungs-Loss {val_loss:.3f}{marker}")

## 5. Hauptlauf: von Grund auf mit der besten Lernrate

Gespeichert wird der Stand mit dem besten Validierungs-Loss. Bei so wenig Text beginnt das Modell irgendwann,
den Trainingstext auswendig zu lernen. Dann steigt der Validierungs-Loss wieder, und spätere Stände wären schlechter.

In [ ]:
scratch = new_scratch_model()
scratch_checkpoint = RUN_DIR / "lotr-scratch-best.pth"
run_training(
    name=f"lotr-scratch-main-lr{best_lr:g}",
    group="lotr-main",
    model=scratch,
    learning_rate=best_lr,
    epochs=CONFIG["main"]["epochs"],
    checkpoint_path=scratch_checkpoint,
)
load_checkpoint(scratch_checkpoint, scratch)
print("Checkpoint:", keep_checkpoint(scratch_checkpoint))

record("LOTR von Grund auf", scratch, CONFIG["scratch_model"]["context_length"])
assert RESULTS["LOTR von Grund auf"]["test_loss"] < RESULTS["Untrainiert"]["test_loss"], (
    "Training hat den Test-Loss nicht gesenkt"
)
del scratch

## 6. GPT-2 original und nachtrainiert

Die OpenAI-Gewichte von GPT-2 Small im Format unseres `GPTModel` (Raschka, *Build a Large Language Model (From Scratch)*).
Geladen wird eine feste Version mit Prüfsumme, einmalig nach `CHECKPOINT_DIR`. Die Datei ist 700 MB groß.

Das Nachtrainieren nutzt denselben Code wie oben, nur mit den OpenAI-Gewichten als Startpunkt und kleiner Lernrate.

In [ ]:
from infm_sota2.model import (
    GPT2_SMALL_WEIGHTS_SHA256,
    GPT2_SMALL_WEIGHTS_URL,
    download_weights,
)

RUN_PRETRAINED = not SMOKE_TEST

if RUN_PRETRAINED:
    weights_path = download_weights(
        GPT2_SMALL_WEIGHTS_URL,
        GPT2_SMALL_WEIGHTS_SHA256,
        CHECKPOINT_DIR / "gpt2-small-124M.pth",
    )

    def new_pretrained_model() -> GPTModel:
        model = GPTModel(GPT2_SMALL_CONFIG)
        state = torch.load(weights_path, map_location="cpu", weights_only=True)
        model.load_state_dict(state)  # strikt: jeder Schlüssel muss passen
        return model.to(DEVICE)

    gpt2 = new_pretrained_model()
    record("GPT-2 original", gpt2, GPT2_SMALL_CONFIG["context_length"])
    del gpt2
else:
    print("SMOKE_TEST: GPT-2 original übersprungen (keine OpenAI-Gewichte geladen)")

In [ ]:
if RUN_PRETRAINED:
    finetuned = new_pretrained_model()
    finetune_checkpoint = RUN_DIR / "gpt2-lotr-finetune-best.pth"
    run_training(
        name=f"gpt2-lotr-finetune-lr{CONFIG['finetune']['learning_rate']:g}",
        group="lotr-finetune",
        model=finetuned,
        learning_rate=CONFIG["finetune"]["learning_rate"],
        epochs=CONFIG["finetune"]["epochs"],
        checkpoint_path=finetune_checkpoint,
    )
    load_checkpoint(finetune_checkpoint, finetuned)
    print("Checkpoint:", keep_checkpoint(finetune_checkpoint))

    record("GPT-2 nachtrainiert", finetuned, GPT2_SMALL_CONFIG["context_length"])
    del finetuned
else:
    print("SMOKE_TEST: Nachtrainieren übersprungen")

## 7. Claude

Claude läuft nicht im Notebook, sondern per Hand in claude.ai:

1. Einen **Inkognito-Chat** öffnen. Der nutzt weder die Memory-Funktion noch frühere Chats.
2. Den Prompt aus der nächsten Zelle unverändert einfügen und die Antwort kopieren (nur die Fortsetzung).
3. Das insgesamt dreimal, jeweils in einem **neuen** Inkognito-Chat.
4. Die drei Antworten in `CLAUDE_COMPLETIONS` eintragen, das im Chat angezeigte Modell in `CLAUDE_MODEL`, dann die Zellen ab hier erneut ausführen.

Zu beachten: Claude kennt Tolkien aus dem eigenen Training und bekommt eine Anweisung statt eines bloßen Satzanfangs.
Einen Loss gibt es für Claude nicht, der Vergleich ist rein qualitativ.

In [ ]:
CLAUDE_PROMPT = (
    "Continue the following text. Reply with the continuation only, "
    "about 30 words.\n\n" + gen_cfg["prompt"]
)
print(CLAUDE_PROMPT)

In [ ]:
# Antworten aus drei neuen Inkognito-Chats, jeweils nur die Fortsetzung
CLAUDE_COMPLETIONS: list[str] = []
CLAUDE_MODEL = ""  # Modellname, wie claude.ai ihn im Chat anzeigt

if CLAUDE_COMPLETIONS:
    assert len(CLAUDE_COMPLETIONS) == gen_cfg["samples"], "Bitte genau drei Antworten"
    assert CLAUDE_MODEL, "Bitte das verwendete Claude-Modell eintragen"
else:
    print("Noch keine Claude-Antworten eingetragen")

## 8. Vergleich

**Test-Loss und Perplexity** sind die belastbaren Zahlen: alle vier GPT-2-Varianten auf demselben Testtext,
mit demselben Tokenizer. Niedriger ist besser. Das Testset wurde weder fürs Training noch für die Auswahl benutzt.

**Kopiert** zeigt für jeden Text die längste Wortfolge, die wörtlich im Trainingstext steht.
Lange Folgen heißen: Das Modell gibt auswendig Gelerntes wieder, statt selbst zu formulieren.
Bei GPT-2 original und Claude kann das auch auf Vorwissen aus deren eigenem Training hindeuten.

Drei Texte pro Modell sind Beispiele, keine Statistik. Und jede Variante lief nur einmal,
kleine Unterschiede im Test-Loss sind deshalb nicht belastbar.

In [ ]:
import pandas as pd

from infm_sota2.training import longest_copied_span

pd.set_option("display.max_colwidth", None)

losses = pd.DataFrame(
    [
        {"Modell": name, "Test-Loss": r["test_loss"], "Perplexity": r["perplexity"]}
        for name, r in RESULTS.items()
    ]
).set_index("Modell")
losses.round(3)

In [ ]:
all_completions = [
    (name, r["completions"]) for name, r in RESULTS.items()
]
if CLAUDE_COMPLETIONS:
    all_completions.append((f"Claude ({CLAUDE_MODEL})", CLAUDE_COMPLETIONS))

rows = []
for name, completions in all_completions:
    for i, completion in enumerate(completions, 1):
        copied = longest_copied_span(completion, train_text)
        rows.append(
            {
                "Modell": name,
                "Nr.": i,
                "Text": gen_cfg["prompt"] + completion,
                "Kopiert (Wörter)": len(copied.split()),
                "Längste kopierte Folge": copied,
            }
        )
pd.DataFrame(rows).set_index(["Modell", "Nr."])

## 9. Fazit

*Nach dem Lauf ausfüllen: Welches Modell hat den niedrigsten Test-Loss? Wie unterscheiden sich die Texte?
Hat unser Modell auswendig gelernt? Was fällt bei Claude auf?*